
# Global Pollution Analysis and Energy Recovery

Dataset-specific implementation using the uploaded dataset.

Objectives:
- Data Cleaning and Feature Engineering
- K-Means Clustering
- Hierarchical Clustering
- Neural Network Regression for Energy Recovery Prediction
- Linear Regression Comparison
- Visualizations and Insights


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from scipy.cluster.hierarchy import linkage, dendrogram

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv('Global_Pollution_Analysis 2.csv')

print(df.shape)
df.head()


## Data Cleaning and Preprocessing

In [ ]:

print(df.isnull().sum())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(exclude='object').columns:
    df[col] = df[col].fillna(df[col].median())

df.info()


## Feature Engineering

In [ ]:

# Pollution Score
df['Total_Pollution_Score'] = (
    df['Air_Pollution_Index'] +
    df['Water_Pollution_Index'] +
    df['Soil_Pollution_Index']
) / 3

# Energy Recovery Efficiency
df['Energy_Recovery_Efficiency'] = (
    df['Energy_Recovered (in GWh)'] /
    (df['Industrial_Waste (in tons)'] + 1)
)

# Label Encoding Country
le = LabelEncoder()
df['Country'] = le.fit_transform(df['Country'])

df.head()


## Yearly Pollution Trend Analysis

In [ ]:

yearly = df.groupby('Year')[['Total_Pollution_Score',
                             'Energy_Recovered (in GWh)']].mean()

yearly.plot(figsize=(10,5))
plt.title('Yearly Pollution and Energy Recovery Trends')
plt.show()


## Feature Scaling

In [ ]:

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)


## K-Means Clustering

In [ ]:

wcss = []

for i in range(1,11):
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    km.fit(scaled_data)
    wcss.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(range(1,11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Clusters')
plt.ylabel('WCSS')
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(scaled_data)

df['KMeans_Cluster'].value_counts()


In [ ]:

plt.figure(figsize=(8,5))
plt.scatter(
    df['Total_Pollution_Score'],
    df['Energy_Recovered (in GWh)'],
    c=df['KMeans_Cluster']
)
plt.xlabel('Pollution Score')
plt.ylabel('Energy Recovered')
plt.title('K-Means Clusters')
plt.show()


## Hierarchical Clustering

In [ ]:

Z = linkage(scaled_data, method='ward')

plt.figure(figsize=(12,5))
dendrogram(Z)
plt.title('Hierarchical Clustering Dendrogram')
plt.show()

hc = AgglomerativeClustering(n_clusters=3)
df['HC_Cluster'] = hc.fit_predict(scaled_data)


## Neural Network Regression

In [ ]:

target = 'Energy_Recovered (in GWh)'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler_x = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test = scaler_x.transform(X_test)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=1
)

pred = model.predict(X_test)

print('Neural Network Results')
print('R2:', r2_score(y_test, pred))
print('MSE:', mean_squared_error(y_test, pred))
print('MAE:', mean_absolute_error(y_test, pred))


In [ ]:

plt.figure(figsize=(7,4))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Training vs Validation Loss')
plt.legend(['Train','Validation'])
plt.show()


## Linear Regression Comparison

In [ ]:

lr = LinearRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print('Linear Regression Results')
print('R2:', r2_score(y_test, pred_lr))
print('MSE:', mean_squared_error(y_test, pred_lr))
print('MAE:', mean_absolute_error(y_test, pred_lr))



# Final Report

## Methodology
1. Cleaned and encoded pollution dataset.
2. Engineered pollution and efficiency metrics.
3. Applied K-Means and Hierarchical Clustering.
4. Built Neural Network and Linear Regression models.
5. Compared prediction performance using R², MSE, and MAE.

## Key Findings
- Countries can be grouped by pollution intensity and recovery capability.
- Pollution indicators strongly influence energy recovery.
- Neural Networks capture nonlinear relationships better than linear models.

## Actionable Recommendations
- Increase renewable energy adoption in high-pollution clusters.
- Improve waste-to-energy conversion technologies.
- Reduce industrial waste generation while maximizing recovery efficiency.
- Use cluster-specific environmental policies for targeted interventions.
